# PV & BESS Hosting Capacity — IEEE 33-Bus MILP
### Improved Accuracy Version

**Key improvements over original notebook:**
1. **Full LinDistFlow (P+Q)** — reactive power and X terms included; voltage drops now accurate for inductive lines.
2. **SOC bounds linearised** — `E[n,t] ∈ [0, Emax[n]]` replaces the bilinear `E >= SOC_min·Emax` form (Var×Var is not LP/MILP-safe in HiGHS). `Emax[n]` is now the **usable** energy window; physical rated capacity = Emax / 0.70.
3. **Simultaneous charge/discharge prevented** — binary `u_ch[n,t]` mutex added (original allowed both at once, inflating apparent capacity).
4. **Reactive load modelled** — Q injections from PV (unity PF → Q_pv = 0) and reactive balance per bus; Q-flow variables added.
5. **Voltage-drop uses both R and X** — `V_j = V_i − R·P − X·Q`; purely resistive drop under-estimates voltage sag on inductive lines (branches 6,7,12,16,19,21,23 have X > R).
6. **Hosting capacity metric added** — `HC_PV = Σ PPVmax_kW` and `HC_BESS_E = Σ Emax_kWh` printed explicitly.
7. **Cycle constraint corrected** — SOC at end of day equals *initial* SOC (not hard-coded 0), so any feasible starting SOC is accepted.
8. **PV curtailment variable added** — distinguishes available PV from dispatched PV for accurate hosting capacity.


## 1. Data — IEEE 33-Bus, Load & PV Profiles

## 1b. Multi-Day Stochastic Profile Generation — Theory & Formulation

This section replaces the original single 24-h load vector with a **7-day stochastic model**
that preserves physical realism while introducing day-to-day variation.
All randomness is seeded for full reproducibility.

---

### Notation

| Symbol | Meaning |
|--------|---------|
| $d \in \{1,\ldots,7\}$ | Day index (Mon–Sun) |
| $t \in \{0,\ldots,23\}$ | Hour index |
| $\lambda^\text{base}[t]$ | Base 24-h normalised load profile (IEEE 33-bus benchmark) |
| $\text{CF}^\text{base}[t]$ | Base 24-h PV capacity factor |
| $\lambda[d,t]$ | Generated load multiplier for day $d$, hour $t$ |
| $\text{CF}[d,t]$ | Generated PV capacity factor for day $d$, hour $t$ |
| $\gamma[d]$ | Day-type scale factor |
| $\delta[d]$ | Peak-hour shift (hours) |

---

### Layer 1 — Day-type scaling

Each day carries a deterministic **daily scale factor** $\gamma[d]$ reflecting
weekday/weekend demand patterns, and a small **peak-hour shift** $\delta[d]$:

$$\lambda_\text{scaled}[d,t] = \gamma[d] \cdot \lambda^\text{base}\!\left[t - \delta[d]\right]$$

| Day | $\gamma[d]$ | $\delta[d]$ | Type |
|-----|------------|------------|------|
| Mon | 1.00 | 0 | Weekday |
| Tue | 1.03 | 0 | Weekday |
| Wed | 1.05 | 0 | Weekday (peak) |
| Thu | 1.02 | 0 | Weekday |
| Fri | 0.98 | −1 | Weekday |
| Sat | 0.86 | +1 | Weekend |
| Sun | 0.80 | +2 | Weekend |

---

### Layer 2 — Correlated slow drift (AR(1) process)

To simulate **temperature-driven demand variation** that is smooth and
correlated across hours, an autoregressive AR(1) process is applied:

$$\varepsilon_\text{slow}[d,t] = \varphi \cdot \varepsilon_\text{slow}[d,t-1]
  + \sigma_\text{slow} \cdot \eta[t], \quad \eta[t] \sim \mathcal{N}(0,1)$$

where $\varphi = 0.75$ (autocorrelation, $\tau \approx 6$ h) and $\sigma_\text{slow} = 0.04$.

---

### Layer 3 — Fast appliance noise

Independent hour-by-hour noise simulating individual appliance switching:

$$\varepsilon_\text{fast}[d,t] \sim \mathcal{N}(0,\, \sigma_\text{fast}^2), \quad \sigma_\text{fast} = 0.013$$

---

### Combined load profile

$$\lambda_\text{raw}[d,t] = \lambda_\text{scaled}[d,t]
  + \varepsilon_\text{slow}[d,t] + \varepsilon_\text{fast}[d,t]$$

A **Gaussian kernel smoother** ($\sigma_k = 1.2$ h) enforces physical continuity:

$$\lambda[d,t] = \frac{\displaystyle\sum_k w_k \cdot \lambda_\text{raw}[d,\, t+k]}
                     {\displaystyle\sum_k w_k}, \quad w_k = e^{-k^2/2\sigma_k^2}$$

Hard bounds: $0.40 \leq \lambda[d,t] \leq 1.15$

---

### PV irradiance model — 3-state weather

Each day is assigned a weather state $s[d]$ drawn from a discrete distribution:

| State | Label | Probability | CF scale $\gamma_s$ | Noise $\sigma_s$ |
|-------|-------|:-----------:|:-------------------:|:----------------:|
| $s=0$ | Clear sky | 0.50 | 1.00 | 0.02 |
| $s=1$ | Partly cloudy | 0.35 | 0.72 | 0.09 |
| $s=2$ | Overcast | 0.15 | 0.38 | 0.15 |

**Clear sky** ($s=0$): smooth multiplicative noise only

$$\text{CF}[d,t] = \text{CF}^\text{base}[t] \cdot \gamma_s \cdot (1 + \sigma_s \cdot \eta[t])$$

**Partly cloudy** ($s=1$): cloud-passing factor $c[t] \sim |\mathcal{N}(0,1)|$ applied

$$\text{CF}[d,t] = \text{CF}^\text{base}[t] \cdot \gamma_s \cdot c[t] \cdot (1 + \sigma_s \cdot \eta[t])$$

**Overcast** ($s=2$): uniform random suppression

$$\text{CF}[d,t] = \text{CF}^\text{base}[t] \cdot \gamma_s \cdot U(0.5,1.0) \cdot (1 + \sigma_s \cdot \eta[t])$$

All CF values clipped to $[0,\,1]$ and smoothed with $\sigma_k = 1.0$ h.

---

### Reproducibility

All random draws use seeded generators:

$$\text{seed}_\text{load}[d] = 42 + d \times 97, \qquad
  \text{seed}_\text{PV}[d]  = 42 + d \times 113 + 500$$

---

### Extension to the MILP (two-stage structure)

The 7-day profiles introduce a **day index** $d \in \mathcal{D} = \{0,\ldots,6\}$.
The MILP naturally separates into two stages:

**Stage 1 — here-and-now (day-independent planning):**

$$x^\text{PV}[n],\; x^B[n],\; P^\text{PV,max}[n],\; P^\text{ch,max}[n],\;
  P^\text{dis,max}[n],\; E^\text{max}[n] \quad \forall n \in \mathcal{N}$$

**Stage 2 — wait-and-see (day-dependent dispatch):**

$$P^\text{pv}[n,d,t],\; P^\text{curt}[n,d,t],\; P^\text{ch}[n,d,t],\;
  P^\text{dis}[n,d,t],\; E[n,d,t]$$
$$P_\ell[d,t],\; Q_\ell[d,t],\; V[n,d,t] \quad \forall n,\ell,d,t$$

The **SOC daily cycle** applies independently per day:

$$E[n,d,23] = E_\text{init}[n,d] \quad \forall n,\; d \in \mathcal{D}$$

This correctly captures that each day starts and ends with the same energy
(the BESS does not carry state across days in this formulation).


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  MULTI-DAY PROFILE GENERATION
#  Produces load_7d[7,24] and pv_7d[7,24] — reproducible via seed.
# ════════════════════════════════════════════════════════════════════
from scipy.ndimage import gaussian_filter1d

SEED_BASE = 42
N_DAYS    = 7

# ── Base profiles (single-day IEEE benchmark) ────────────────────
LOAD_BASE = np.array([
    0.685,0.659,0.644,0.644,0.674,0.731,
    0.768,0.830,0.882,0.916,0.959,0.989,
    0.987,0.974,0.968,0.965,0.950,0.929,
    0.990,1.000,0.958,0.887,0.804,0.735,
])
PV_BASE = np.array([
    0,0,0,0,0,0,
    0.10090,0.32375,0.56753,0.75722,0.91625,0.98658,
    0.75912,0.73428,0.56403,0.41945,0.21907,0.05410,
    0,0,0,0,0,0,
])

# ── Day configuration: (name, type, gamma, peak_shift) ──────────
DAY_CONFIG = [
    ('Monday',    'Weekday', 1.00,  0),
    ('Tuesday',   'Weekday', 1.03,  0),
    ('Wednesday', 'Weekday', 1.05,  0),
    ('Thursday',  'Weekday', 1.02,  0),
    ('Friday',    'Weekday', 0.98, -1),
    ('Saturday',  'Weekend', 0.86,  1),
    ('Sunday',    'Weekend', 0.80,  2),
]

# ── Weather configuration ────────────────────────────────────────
WEATHER = {
    'names' : ['Clear sky', 'Partly cloudy', 'Overcast'],
    'probs' : [0.50,          0.35,            0.15],
    'scales': [1.00,          0.72,            0.38],
    'noise' : [0.02,          0.09,            0.15],
    'labels': ['[CLR]',       '[CLD]',         '[OVC]'],
}


def _generate_load(day_idx):
    """3-layer stochastic load model for one day. Returns shape (24,)."""
    _, _, gamma, shift = DAY_CONFIG[day_idx]
    rng = np.random.default_rng(SEED_BASE + day_idx * 97)

    # Layer 1 — day-type scaling + peak-hour shift
    shifted = np.roll(LOAD_BASE, shift) * gamma

    # Layer 2 — AR(1) slow drift  (phi=0.75, sigma=0.04)
    eps_slow = np.zeros(24)
    for t in range(1, 24):
        eps_slow[t] = 0.75 * eps_slow[t-1] + 0.04 * rng.standard_normal()

    # Layer 3 — fast appliance noise
    eps_fast = rng.standard_normal(24) * 0.013

    raw = shifted + eps_slow + eps_fast
    smoothed = gaussian_filter1d(raw, sigma=1.2, mode='nearest')
    return np.clip(smoothed, 0.40, 1.15)


def _generate_pv(day_idx):
    """3-state weather PV model for one day. Returns (cf array (24,), weather_idx)."""
    rng = np.random.default_rng(SEED_BASE + day_idx * 113 + 500)

    w = rng.choice(3, p=WEATHER['probs'])
    scale, noise_s = WEATHER['scales'][w], WEATHER['noise'][w]

    if w == 0:   # Clear sky
        cf_raw = PV_BASE * scale * (1 + rng.standard_normal(24) * noise_s)
    elif w == 1: # Partly cloudy
        cloud  = np.clip(1 - np.abs(rng.standard_normal(24)) * 0.3, 0.3, 1.0)
        cf_raw = PV_BASE * scale * cloud * (1 + rng.standard_normal(24) * noise_s)
    else:        # Overcast
        supp   = rng.uniform(0.5, 1.0, 24)
        cf_raw = PV_BASE * scale * supp * (1 + rng.standard_normal(24) * noise_s)

    cf_raw    = np.maximum(cf_raw, 0)
    cf_smooth = gaussian_filter1d(cf_raw, sigma=1.0, mode='nearest')
    return np.clip(cf_smooth, 0.0, 1.0), w


def generate_week():
    """Generate all 7 days. Returns dict with arrays of shape (7, 24)."""
    load_arr    = np.zeros((N_DAYS, 24))
    pv_arr      = np.zeros((N_DAYS, 24))
    weather_arr = np.zeros(N_DAYS, dtype=int)
    for d in range(N_DAYS):
        load_arr[d], (pv_arr[d], weather_arr[d]) = (
            _generate_load(d), _generate_pv(d)
        )
    return {'load': load_arr, 'pv': pv_arr, 'weather': weather_arr}


# ── Generate ─────────────────────────────────────────────────────
week      = generate_week()
load_7d   = week['load']    # shape (7, 24)  — load multiplier per day & hour
pv_7d     = week['pv']      # shape (7, 24)  — PV capacity factor per day & hour
weather_7d = week['weather'] # shape (7,)     — weather state index per day

D      = list(range(N_DAYS))   # day index set for Pyomo
T24    = list(range(24))        # hour index set (alias)

print("7-day profiles generated  (seed = 42, fully reproducible)")
print(f"  load_7d  shape: {load_7d.shape}   range: [{load_7d.min():.3f}, {load_7d.max():.3f}]")
print(f"  pv_7d    shape: {pv_7d.shape}   range: [{pv_7d.min():.3f}, {pv_7d.max():.3f}]")
print()
print(f"  {'Day':<12} {'Type':<8} {'gamma':>5}  {'Weather':<14} {'Peak CF':>7}  {'PV Energy':>9}")
print(f"  {'─'*12} {'─'*8} {'─'*5}  {'─'*14} {'─'*7}  {'─'*9}")
for d in range(N_DAYS):
    name, dtype, gamma, _ = DAY_CONFIG[d]
    w  = weather_7d[d]
    print(f"  {name:<12} {dtype:<8} {gamma:>5.2f}  "
          f"{WEATHER['labels'][w]} {WEATHER['names'][w]:<10} "
          f"{pv_7d[d].max():>7.3f}  {pv_7d[d].sum():>9.2f} pu-h")


In [ ]:
# ════════════════════════════════════════════════════════════════════
#  VISUALIZATION — 7-day load & PV profiles (IEEE publication style)
# ════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

plt.rcParams.update({
    'font.family'     : 'Times New Roman',
    'font.size'       : 9,
    'axes.linewidth'  : 0.7,
    'axes.grid'       : True,
    'grid.linewidth'  : 0.4,
    'grid.alpha'      : 0.35,
    'grid.linestyle'  : '--',
    'xtick.major.size': 3,
    'ytick.major.size': 3,
    'figure.dpi'      : 130,
})

NAVY  = '#1A3055'
GRAY  = '#8A8A8A'
HOURS = np.arange(24)
XTICKS      = [0, 6, 12, 18, 23]
XLABELS     = ['0', '6', '12', '18', '23']
LOAD_COLORS = ['#1A4A7A','#2E6FA3','#3A8FC4','#2E85A0',
               '#1D6B88','#4A90A4','#6AAABA']
PV_COLORS   = ['#A85C00','#C8820A','#E09A20','#C87A10',
               '#B06A05','#D08A18','#E8A030']
W_COLORS    = ['#E8A020', '#5A8FC0', '#888880']

fig = plt.figure(figsize=(14, 10))
fig.patch.set_facecolor('#FAFAFA')
outer = gridspec.GridSpec(4, 1, figure=fig, hspace=0.52,
                          top=0.93, bottom=0.07, left=0.06, right=0.97)
gs0 = gridspec.GridSpecFromSubplotSpec(1, 7, subplot_spec=outer[0], wspace=0.12)
gs1 = gridspec.GridSpecFromSubplotSpec(1, 7, subplot_spec=outer[1], wspace=0.12)

for d in range(N_DAYS):
    name, dtype, gamma, _ = DAY_CONFIG[d]
    w_lbl  = WEATHER['labels'][weather_7d[d]]
    w_name = WEATHER['names'][weather_7d[d]]
    w_col  = W_COLORS[weather_7d[d]]
    t_col  = NAVY if dtype == 'Weekday' else '#8B0000'

    # ── load small multiple
    ax_l = fig.add_subplot(gs0[d])
    ax_l.set_facecolor('#FAFAFA')
    ax_l.fill_between(HOURS, LOAD_BASE, alpha=0.12, color=GRAY)
    ax_l.plot(HOURS, LOAD_BASE, color=GRAY, lw=0.9, ls='--')
    ax_l.plot(HOURS, load_7d[d], color=LOAD_COLORS[d], lw=1.5,
              marker='o', ms=2.5, markevery=4)
    ax_l.fill_between(HOURS, load_7d[d], alpha=0.10, color=LOAD_COLORS[d])
    ax_l.set_xlim(-0.5, 23.5); ax_l.set_ylim(0.38, 1.22)
    ax_l.set_xticks(XTICKS); ax_l.set_xticklabels(XLABELS, fontsize=7)
    if d == 0:
        ax_l.set_yticks([0.4, 0.6, 0.8, 1.0])
        ax_l.set_yticklabels(['0.4','0.6','0.8','1.0'], fontsize=7)
        ax_l.set_ylabel('Load (pu)', fontsize=8)
    else:
        ax_l.set_yticklabels([])
    ax_l.set_title(f'{name[:3]}\n({dtype})', fontsize=8, fontweight='bold',
                   color=t_col, pad=3)
    ax_l.text(0.97, 0.95, f'g={gamma:.2f}', transform=ax_l.transAxes,
              fontsize=7, color='#2E6FA3', ha='right', va='top')

    # ── PV small multiple
    ax_p = fig.add_subplot(gs1[d])
    ax_p.set_facecolor('#FAFAFA')
    ax_p.fill_between(HOURS, PV_BASE, alpha=0.12, color=GRAY)
    ax_p.plot(HOURS, PV_BASE, color=GRAY, lw=0.9, ls='--')
    ax_p.plot(HOURS, pv_7d[d], color=PV_COLORS[d], lw=1.5,
              marker='s', ms=2.5, markevery=4)
    ax_p.fill_between(HOURS, pv_7d[d], alpha=0.14, color=PV_COLORS[d])
    ax_p.set_xlim(-0.5, 23.5); ax_p.set_ylim(-0.05, 1.12)
    ax_p.set_xticks(XTICKS); ax_p.set_xticklabels(XLABELS, fontsize=7)
    if d == 0:
        ax_p.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax_p.set_yticklabels(['0','0.25','0.5','0.75','1.0'], fontsize=7)
        ax_p.set_ylabel('CF (pu)', fontsize=8)
    else:
        ax_p.set_yticklabels([])
    ax_p.set_title(f'{w_lbl} {w_name}', fontsize=7.5,
                   color=w_col, pad=3, fontweight='bold')
    ax_p.text(0.97, 0.95, f'E={pv_7d[d].sum():.1f}',
              transform=ax_p.transAxes, fontsize=6.5,
              color='#C8820A', ha='right', va='top')

# ── Load overlay
ax_lo = fig.add_subplot(outer[2])
ax_lo.set_facecolor('#FAFAFA')
ax_lo.plot(HOURS, LOAD_BASE, color=GRAY, lw=1.2, ls='--', label='Base')
for d in range(N_DAYS):
    ls = '-' if DAY_CONFIG[d][1] == 'Weekday' else '-.'
    ax_lo.plot(HOURS, load_7d[d], color=LOAD_COLORS[d], lw=1.3,
               ls=ls, alpha=0.85,
               label=f'{DAY_CONFIG[d][0][:3]} (g={DAY_CONFIG[d][2]:.2f})')
ax_lo.axhline(1.0, color='#C0392B', lw=0.7, ls=':', alpha=0.7)
ax_lo.set_xlim(-0.5, 23.5); ax_lo.set_ylim(0.38, 1.22)
ax_lo.set_xticks(range(0, 24, 2))
ax_lo.set_xticklabels([str(h) for h in range(0, 24, 2)], fontsize=8)
ax_lo.set_ylabel('Normalised load (pu)', fontsize=9)
ax_lo.set_xlabel('Hour of day', fontsize=9)
ax_lo.legend(loc='upper left', ncol=4, fontsize=7.5,
             framealpha=0.85, edgecolor='#CCCCCC',
             handlelength=1.6, columnspacing=0.8)
ax_lo.set_title('(c)  All-day load overlay - 7 days',
                fontsize=9, fontweight='bold', color=NAVY, loc='left', pad=4)

# ── PV overlay
ax_po = fig.add_subplot(outer[3])
ax_po.set_facecolor('#FAFAFA')
ax_po.plot(HOURS, PV_BASE, color=GRAY, lw=1.2, ls='--', label='Base')
for d in range(N_DAYS):
    ax_po.plot(HOURS, pv_7d[d], color=PV_COLORS[d], lw=1.3, alpha=0.85,
               label=f'{DAY_CONFIG[d][0][:3]} ({WEATHER["names"][weather_7d[d]]})')
ax_po.set_xlim(-0.5, 23.5); ax_po.set_ylim(-0.03, 1.12)
ax_po.set_xticks(range(0, 24, 2))
ax_po.set_xticklabels([str(h) for h in range(0, 24, 2)], fontsize=8)
ax_po.set_ylabel('PV capacity factor (pu)', fontsize=9)
ax_po.set_xlabel('Hour of day', fontsize=9)
ax_po.legend(loc='upper left', ncol=4, fontsize=7.5,
             framealpha=0.85, edgecolor='#CCCCCC',
             handlelength=1.6, columnspacing=0.8)
ax_po.set_title('(d)  PV capacity factor overlay - 7 days (weather state per day)',
                fontsize=9, fontweight='bold', color=NAVY, loc='left', pad=4)

# ── Row labels & suptitle
fig.text(0.005, 0.955, '(a)  Daily load profiles',
         fontsize=9, fontweight='bold', color=NAVY, va='top')
fig.text(0.005, 0.715, '(b)  Daily PV capacity factor',
         fontsize=9, fontweight='bold', color=NAVY, va='top')

weather_handles = [
    Line2D([0],[0], color=W_COLORS[i], lw=2,
           label=f'{WEATHER["labels"][i]} {WEATHER["names"][i]} (p={WEATHER["probs"][i]:.2f})')
    for i in range(3)
]
fig.legend(handles=weather_handles, loc='lower right', fontsize=7.5,
           framealpha=0.9, edgecolor='#CCCCCC',
           title='Weather states', title_fontsize=8,
           bbox_to_anchor=(0.97, 0.01))

fig.suptitle(
    'IEEE 33-Bus — 7-Day Stochastic Load & PV Profiles\n'
    'AR(1) Load Model | 3-State Weather PV Model | Seed = 42',
    fontsize=11, fontweight='bold', color=NAVY, y=0.985
)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig('fig1_weekly_profiles.png', dpi=200, bbox_inches='tight',
            facecolor='#FAFAFA', edgecolor='none')
plt.show()
print("Figure saved: fig1_weekly_profiles.png")


## 1c. MILP Extension — Adding the Day Index $d$

The 7-day profiles extend the MILP by adding $d \in \mathcal{D}$ to all
**operational** variables while keeping **planning** variables day-independent.

### Variable changes

| Variable | Original shape | Extended shape | Note |
|----------|---------------|---------------|------|
| `lmult[t]` | Param (24,) | Param (7, 24) | per-day load multiplier |
| `CF[t]` | Param (24,) | Param (7, 24) | per-day PV capacity factor |
| `Ppv[n,t]` | (33, 24) | (33, 7, 24) | per-day dispatch |
| `Pcurt[n,t]` | (33, 24) | (33, 7, 24) | per-day curtailment |
| `Pch[n,t]` | (33, 24) | (33, 7, 24) | per-day charge |
| `Pdis[n,t]` | (33, 24) | (33, 7, 24) | per-day discharge |
| `E[n,t]` | (33, 24) | (33, 7, 24) | per-day SOC |
| `E_init[n]` | (33,) | (33, 7) | per-day initial SOC |
| `Pflow[l,t]` | (32, 24) | (32, 7, 24) | per-day branch flow |
| `Qflow[l,t]` | (32, 24) | (32, 7, 24) | per-day reactive flow |
| `V[n,t]` | (33, 24) | (33, 7, 24) | per-day bus voltage |

### Model size after extension

| Quantity | Single day | 7 days | Factor |
|----------|-----------|--------|--------|
| Binary vars | 66 | 66 | ×1 (unchanged) |
| Continuous vars | ~7,245 | ~50,482 | ×7 |
| Constraints | ~10,400 | ~72,800 | ×7 |

The model remains an **MILP** — no new binary variables are added.
HiGHS handles this scale comfortably (typically 5–30 s solve time).


In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary, Reals,
    Constraint, Objective, minimize, value
)
from pyomo.opt import SolverFactory, TerminationCondition

# ══════════════════════════════════════════════════════════════════
#  BASE VALUES  —  IEEE 33-bus (Baran-Wu 1989)
#  Vbase = 12.66 kV (L-L),  Sbase = 10 MVA  →  Zbase = 16.03 Ω
#  The branch R,X values in this dataset are in OHMS.
#  We normalise them to pu here so that LinDistFlow is dimensionally
#  consistent:   V_j = V_i  −  R_pu · P_pu  −  X_pu · Q_pu
# ══════════════════════════════════════════════════════════════════
Vbase_kV   = 12.66          # kV  (line-to-line)
Sbase_kW   = 10_000.0       # kW  (= 10 MVA)
Sbase_kVAr = 10_000.0
Zbase_ohm  = (Vbase_kV**2 * 1e6) / (Sbase_kW * 1e3)   # = 16.03 Ω

print(f"Zbase = {Zbase_ohm:.4f} Ω   (Vbase={Vbase_kV} kV, Sbase={Sbase_kW} kW)")

nb   = 33
slack = 1

# ─── Bus loads (kW, kVAr) ────────────────────────────────────────
bus_load = {
    1: (0,0),     2: (100,60),   3: (90,40),   4: (120,80),
    5: (60,30),   6: (60,20),    7: (200,100),  8: (200,100),
    9: (60,20),  10: (60,20),   11: (45,30),   12: (60,35),
   13: (60,35),  14: (120,80),  15: (60,10),   16: (60,20),
   17: (60,20),  18: (90,40),   19: (90,40),   20: (90,40),
   21: (90,40),  22: (90,40),   23: (90,50),   24: (420,200),
   25: (420,200),26: (60,25),   27: (60,25),   28: (60,20),
   29: (120,70), 30: (200,600), 31: (150,70),  32: (210,100),
   33: (60,40),
}

# ─── Branches: (from, to, R_ohm, X_ohm) — converted to pu below ─
branches_raw = {
     1:(1, 2, 0.0922,0.0470),   2:(2, 3, 0.4930,0.2511),
     3:(3, 4, 0.3660,0.1864),   4:(4, 5, 0.3811,0.1941),
     5:(5, 6, 0.8190,0.7070),   6:(6, 7, 0.1872,0.6188),
     7:(7, 8, 1.7114,1.2351),   8:(8, 9, 1.0300,0.7400),
     9:(9,10, 1.0440,0.7400),  10:(10,11,0.1966,0.0650),
    11:(11,12,0.3744,0.1238),  12:(12,13,1.4680,1.1550),
    13:(13,14,0.5416,0.7129),  14:(14,15,0.5910,0.5260),
    15:(15,16,0.7463,0.5450),  16:(16,17,1.2890,1.7210),
    17:(17,18,0.7320,0.5740),  18:(2, 19,0.1640,0.1565),
    19:(19,20,1.5042,1.3554),  20:(20,21,0.4095,0.4784),
    21:(21,22,0.7089,0.9373),  22:(3, 23,0.4512,0.3083),
    23:(23,24,0.8980,0.7091),  24:(24,25,0.8960,0.7011),
    25:(6, 26,0.2030,0.1034),  26:(26,27,0.2842,0.1447),
    27:(27,28,1.0590,0.9337),  28:(28,29,0.8042,0.7006),
    29:(29,30,0.5075,0.2585),  30:(30,31,0.9744,0.9630),
    31:(31,32,0.3105,0.3619),  32:(32,33,0.3410,0.5302),
}

# Convert to per-unit
branches = {
    l: (fi, ti, r/Zbase_ohm, x/Zbase_ohm)
    for l, (fi, ti, r, x) in branches_raw.items()
}

# ─── 24-hour profiles ────────────────────────────────────────────
T = list(range(24))
load_mult = np.array([
    0.685,0.659,0.644,0.644,0.674,0.731,
    0.768,0.830,0.882,0.916,0.959,0.989,
    0.987,0.974,0.968,0.965,0.950,0.929,
    0.990,1.000,0.958,0.887,0.804,0.735,
], dtype=float)
pv_cf = np.array([
    0,0,0,0,0,0,
    0.10090,0.32375,0.56753,0.75722,0.91625,0.98658,
    0.75912,0.73428,0.56403,0.41945,0.21907,0.05410,
    0,0,0,0,0,0,
], dtype=float)
dt = 1.0  # hours

# ─── Adjacency (precomputed — never use value() inside constraints) ──
from_bus  = {l: branches[l][0] for l in branches}
to_bus    = {l: branches[l][1] for l in branches}
in_lines  = {n: [l for l in branches if to_bus[l]  == n] for n in range(1, nb+1)}
out_lines = {n: [l for l in branches if from_bus[l] == n] for n in range(1, nb+1)}

# ─── Quick sanity check: base-case min voltage ───────────────────
from collections import deque
_ch={n:[] for n in range(1,nb+1)}; _pb={}
for l,(fi,ti,r,x) in branches.items(): _ch[fi].append(ti); _pb[ti]=l
_bfs=[1]; _q=deque([1])
while _q:
    _node=_q.popleft()
    for c in _ch[_node]: _bfs.append(c); _q.append(c)
_P={l:0.0 for l in branches}; _Q={l:0.0 for l in branches}
for _node in reversed(_bfs[1:]):
    l=_pb[_node]; fi,ti,r,x=branches[l]
    _out=[ll for ll,(fii,tii,*_) in branches.items() if fii==_node]
    _P[l]=bus_load[_node][0]/Sbase_kW+sum(_P[ll] for ll in _out)
    _Q[l]=bus_load[_node][1]/Sbase_kVAr+sum(_Q[ll] for ll in _out)
_V={1:1.0}
for _node in _bfs[1:]:
    l=_pb[_node]; fi,ti,r,x=branches[l]
    _V[_node]=_V[fi]-r*_P[l]-x*_Q[l]
_minV=min(_V.values()); _minbus=min(_V,key=_V.get)
print(f"Base-case min voltage (no DER, peak load): {_minV:.4f} pu  at bus {_minbus}")
print(f"  Literature reference (Baran-Wu 1989): ~0.9131 pu at bus 18")
print(f"  Max Q-flow: {max(_Q.values()):.4f} pu = {max(_Q.values())*Sbase_kVAr:.0f} kVAr")
print("Data loaded OK.")


## 2. MILP Formulation — Full LinDistFlow (P + Q)

**Voltage-drop equation (per branch l, hour t):**

$$V_j(t) = V_i(t) - R_l \cdot P_l(t) - X_l \cdot Q_l(t)$$

This replaces the P-only approximation `V_j = V_i − R·P` in the original,
which underestimates voltage sag on branches with significant reactance (e.g., branches 6, 7, 12, 16).

**SOC dynamics (fix #2 & #3):**

$$E_{n,t} = E_{n,t-1} + \eta_c P^{ch}_{n,t} \Delta t - \frac{P^{dis}_{n,t}}{\eta_d} \Delta t$$

$$0.20 \cdot E^{max}_n \le E_{n,t} \le 0.90 \cdot E^{max}_n$$

$$u^{ch}_{n,t} + u^{dis}_{n,t} \le 1 \quad \text{(mutex)}$$


In [ ]:
# ─── Planning limits ──────────────────────────
Npv_max   = 5
Nbess_max = 3

PV_cap_max_kW  = 5000.0    # per-site max (kW)
BESS_p_max_kW  = 2000.0    # per-site charge/discharge max (kW)
BESS_e_max_kWh = 5000.0    # per-site energy capacity max (kWh)

Vmin = 0.90;  Vmax = 1.05;  Vref = 1.0
Pline_max_pu = 1.5
Qline_max_pu = 1.5

SOC_min_frac = 0.20
SOC_max_frac = 0.90
eta_ch  = 0.95
eta_dis = 0.95
dt      = 1.0   # hours

# pu conversions
PV_cap_max_pu = PV_cap_max_kW  / Sbase_kW
P_cap_max_pu  = BESS_p_max_kW  / Sbase_kW
E_cap_max_pu  = BESS_e_max_kWh / Sbase_kW

# ─── Build model ──────────────────────────────
m = ConcreteModel()

# Sets
m.N = Set(initialize=list(range(1, nb+1)))   # buses 1..33
m.L = Set(initialize=list(branches.keys()))  # branches 1..32
m.T = Set(initialize=T24)                    # hours 0..23
m.D = Set(initialize=D)                      # days 0..6  ← NEW

# Network params (time-invariant)
m.Pd  = Param(m.N, initialize={i: bus_load[i][0]/Sbase_kW   for i in range(1,nb+1)}, within=NonNegativeReals)
m.Qd  = Param(m.N, initialize={i: bus_load[i][1]/Sbase_kVAr for i in range(1,nb+1)}, within=NonNegativeReals)
m.R   = Param(m.L, initialize={l: branches[l][2] for l in branches})
m.X   = Param(m.L, initialize={l: branches[l][3] for l in branches})
m.dt  = Param(initialize=dt)

# Multi-day profiles: lmult[d,t] and CF[d,t]  ← UPDATED
m.lmult = Param(m.D, m.T, initialize={(d,t): float(load_7d[d,t]) for d in D for t in T24})
m.CF    = Param(m.D, m.T, initialize={(d,t): float(pv_7d[d,t])   for d in D for t in T24})

# ── Planning variables (day-independent — same hardware for all 7 days) ──
m.xPV     = Var(m.N, within=Binary)
m.xB      = Var(m.N, within=Binary)
m.PPVmax  = Var(m.N, within=NonNegativeReals)
m.Pchmax  = Var(m.N, within=NonNegativeReals)
m.Pdismax = Var(m.N, within=NonNegativeReals)
m.Emax    = Var(m.N, within=NonNegativeReals)

# ── Operational variables (per day)  ← UPDATED: all indexed by (N, D, T) ──
m.Ppv   = Var(m.N, m.D, m.T, within=NonNegativeReals)
m.Pcurt = Var(m.N, m.D, m.T, within=NonNegativeReals)
m.Pch   = Var(m.N, m.D, m.T, within=NonNegativeReals)
m.Pdis  = Var(m.N, m.D, m.T, within=NonNegativeReals)
m.E     = Var(m.N, m.D, m.T, within=NonNegativeReals)
m.LS    = Var(m.N, m.D, m.T, within=NonNegativeReals)

# ── Network variables (per day) ──
m.Pflow = Var(m.L, m.D, m.T, within=Reals)
m.Qflow = Var(m.L, m.D, m.T, within=Reals)
m.V     = Var(m.N, m.D, m.T, within=Reals)

# ── Initial energy per day (linear aux — avoids Var x Var) ──
m.E_init = Var(m.N, m.D, within=NonNegativeReals)

n_binary  = 2 * nb
n_cont_plan = 4 * nb
n_cont_op = nb * N_DAYS * 24 * 6   # Ppv,Pcurt,Pch,Pdis,E,LS
n_network = 32 * N_DAYS * 24 * 2 + nb * N_DAYS * 24  # Pflow,Qflow,V
n_einit   = nb * N_DAYS

print(f"Model built  |  Sbase={Sbase_kW} kW  Zbase={Zbase_ohm:.2f} ohm")
print(f"  Days: {N_DAYS}  |  Hours/day: 24  |  Buses: {nb}  |  Branches: 32")
print(f"  Binary vars      : {n_binary}")
print(f"  Planning vars    : {n_cont_plan}")
print(f"  Operational vars : {n_cont_op}")
print(f"  Network vars     : {n_network}")
print(f"  E_init vars      : {n_einit}")
print(f"  TOTAL vars       : {n_binary+n_cont_plan+n_cont_op+n_network+n_einit}")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONSTRAINTS  —  all linear (MILP-safe)
#  Day index d added to all operational constraints.
#  Planning constraints (installation links) remain day-independent.
# ═══════════════════════════════════════════════════════════════════

# ── Installation links (big-M, day-independent) ─────────────────
m.c_pv_link = Constraint(m.N, rule=lambda m,n: m.PPVmax[n]  <= PV_cap_max_pu * m.xPV[n])
m.c_b1      = Constraint(m.N, rule=lambda m,n: m.Pchmax[n]  <= P_cap_max_pu  * m.xB[n])
m.c_b2      = Constraint(m.N, rule=lambda m,n: m.Pdismax[n] <= P_cap_max_pu  * m.xB[n])
m.c_b3      = Constraint(m.N, rule=lambda m,n: m.Emax[n]    <= E_cap_max_pu  * m.xB[n])

m.c_npv = Constraint(expr=sum(m.xPV[n] for n in m.N) <= Npv_max)
m.c_nb  = Constraint(expr=sum(m.xB[n]  for n in m.N) <= Nbess_max)

# ── PV: dispatched + curtailed = available CF x capacity  (per day) ──
m.c_pv_avail = Constraint(m.N, m.D, m.T,
    rule=lambda m,n,d,t: m.Ppv[n,d,t] + m.Pcurt[n,d,t] == m.CF[d,t] * m.PPVmax[n])

# ── BESS power bounds (per day) ──────────────────────────────────
m.c_ch_cap  = Constraint(m.N, m.D, m.T, rule=lambda m,n,d,t: m.Pch[n,d,t]  <= m.Pchmax[n])
m.c_dis_cap = Constraint(m.N, m.D, m.T, rule=lambda m,n,d,t: m.Pdis[n,d,t] <= m.Pdismax[n])

# ── SOC dynamics (per day) ───────────────────────────────────────
# E[n,d,0] uses E_init[n,d]; subsequent hours use previous hour E.
# E_init[n,d] is a free variable — optimizer picks best starting SOC each day.
def soc_dyn(m, n, d, t):
    if t == 0:
        return m.E[n,d,t] == m.E_init[n,d] + (eta_ch*m.Pch[n,d,t] - (1/eta_dis)*m.Pdis[n,d,t])*m.dt
    return m.E[n,d,t] == m.E[n,d,t-1] + (eta_ch*m.Pch[n,d,t] - (1/eta_dis)*m.Pdis[n,d,t])*m.dt
m.c_soc = Constraint(m.N, m.D, m.T, rule=soc_dyn)

# SOC bounds: E in [0, Emax]  — LINEAR (Emax is USABLE capacity)
# Physical rated = Emax / (SOC_max - SOC_min) = Emax / 0.70
m.c_soc_hi   = Constraint(m.N, m.D, m.T, rule=lambda m,n,d,t: m.E[n,d,t]   <= m.Emax[n])
m.c_einit_hi = Constraint(m.N, m.D,      rule=lambda m,n,d:   m.E_init[n,d] <= m.Emax[n])

# Daily cycle: battery returns to initial SOC at end of each day
m.c_cycle = Constraint(m.N, m.D, rule=lambda m,n,d: m.E[n,d,23] == m.E_init[n,d])

# ── Voltage constraints (per day) ────────────────────────────────
m.c_v_lo  = Constraint(m.N, m.D, m.T, rule=lambda m,n,d,t: m.V[n,d,t] >= Vmin)
m.c_v_hi  = Constraint(m.N, m.D, m.T, rule=lambda m,n,d,t: m.V[n,d,t] <= Vmax)
m.c_slack = Constraint(m.D, m.T,      rule=lambda m,d,t:   m.V[slack,d,t] == Vref)

# LinDistFlow voltage drop: V_j = V_i - R*P - X*Q  (per day)
def vdrop(m, l, d, t):
    i = from_bus[l]; j = to_bus[l]
    return m.V[j,d,t] == m.V[i,d,t] - m.R[l]*m.Pflow[l,d,t] - m.X[l]*m.Qflow[l,d,t]
m.c_vdrop = Constraint(m.L, m.D, m.T, rule=vdrop)

# Line flow limits (per day)
m.c_plim = Constraint(m.L, m.D, m.T, rule=lambda m,l,d,t: (-Pline_max_pu, m.Pflow[l,d,t], Pline_max_pu))
m.c_qlim = Constraint(m.L, m.D, m.T, rule=lambda m,l,d,t: (-Qline_max_pu, m.Qflow[l,d,t], Qline_max_pu))

# ── Active power balance (per day) ──────────────────────────────
# Slack bus excluded — substation supplies whatever P is needed.
def p_balance(m, n, d, t):
    if n == slack:
        return Constraint.Skip
    Pin  = sum(m.Pflow[l,d,t] for l in in_lines[n])
    Pout = sum(m.Pflow[l,d,t] for l in out_lines[n])
    dem  = m.Pd[n] * m.lmult[d,t]
    return Pin - Pout == -(dem - m.LS[n,d,t] - m.Ppv[n,d,t] - m.Pdis[n,d,t] + m.Pch[n,d,t])
m.c_pbal = Constraint(m.N, m.D, m.T, rule=p_balance)

# ── Reactive power balance (per day) ────────────────────────────
# Slack bus excluded. PV at unity PF → no Q injection from PV.
def q_balance(m, n, d, t):
    if n == slack:
        return Constraint.Skip
    Qin  = sum(m.Qflow[l,d,t] for l in in_lines[n])
    Qout = sum(m.Qflow[l,d,t] for l in out_lines[n])
    return Qin - Qout == -m.Qd[n] * m.lmult[d,t]
m.c_qbal = Constraint(m.N, m.D, m.T, rule=q_balance)

# ── Load shedding bound (per day) ────────────────────────────────
m.c_ls = Constraint(m.N, m.D, m.T,
    rule=lambda m,n,d,t: m.LS[n,d,t] <= m.Pd[n]*m.lmult[d,t])

# ── Objective: minimise total ENS across all 7 days ─────────────
m.obj = Objective(
    expr=sum(m.LS[n,d,t]*m.dt for n in m.N for d in m.D for t in m.T),
    sense=minimize
)

print("All constraints built — fully linear (MILP-safe).")
print("  Slack bus excluded from P/Q balance (grid = free source/sink).")
print("  Emax[n] = usable energy (physical rated = Emax / 0.70).")
print(f"  SOC cycle applied per day: E[n,d,23] = E_init[n,d]")


## 3. Solve

In [ ]:
def pick_solver():
    for name in ["cbc", "highs", "gurobi", "glpk"]:
        try:
            s = SolverFactory(name)
            if s.available():
                return name
        except Exception:
            pass
    return None

solver_name = pick_solver()
if solver_name is None:
    raise RuntimeError("No MILP solver found. Install: conda install -c conda-forge coincbc")
print("Using solver:", solver_name)

opt = SolverFactory(solver_name)
res = opt.solve(m, tee=True, load_solutions=False)

from pyomo.opt import TerminationCondition as TC
tc = res.solver.termination_condition
print("\nTermination condition:", tc)

if tc in [TC.optimal, TC.feasible]:
    m.solutions.load_from(res)
    total_ens = value(m.obj)*Sbase_kW
    print(f"Solved.  Total ENS = {total_ens:.2f} kWh  ({total_ens/N_DAYS:.2f} kWh/day avg over {N_DAYS} days)")
else:
    print("\n✗ INFEASIBLE — running base-case voltage check...")
    # Quick diagnosis: compute base-case min voltage
    from collections import deque
    import numpy as np
    _ch={n:[] for n in range(1,nb+1)}; _pb={}
    for l,(fi,ti,r,x) in branches.items(): _ch[fi].append(ti); _pb[ti]=l
    _bfs=[1]; _q=deque([1])
    while _q:
        _node=_q.popleft()
        for c in _ch[_node]: _bfs.append(c); _q.append(c)
    _P={l:0.0 for l in branches}; _Q={l:0.0 for l in branches}
    for _node in reversed(_bfs[1:]):
        l=_pb[_node]; fi,ti,r,x=branches[l]
        _out=[ll for ll,(fii,tii,*_) in branches.items() if fii==_node]
        _P[l]=bus_load[_node][0]/Sbase_kW+sum(_P[ll] for ll in _out)
        _Q[l]=bus_load[_node][1]/Sbase_kVAr+sum(_Q[ll] for ll in _out)
    _V={1:1.0}
    for _node in _bfs[1:]:
        l=_pb[_node]; fi,ti,r,x=branches[l]
        _V[_node]=_V[fi]-r*_P[l]-x*_Q[l]
    minV=min(_V.values()); minbus=min(_V,key=_V.get)
    maxQ=max(_Q.values())
    print(f"  Base-case min voltage: {minV:.4f} pu at bus {minbus}  (Vmin limit={Vmin})")
    print(f"  Base-case max Q-flow:  {maxQ:.4f} pu  (Qline_max_pu={Qline_max_pu})")
    if minV < Vmin:
        print(f"  → Network violates Vmin={Vmin} BEFORE any DER is placed.")
        print(f"    The optimizer cannot fix this with ≤{Npv_max} PV sites.")
        print(f"    Try: set Vmin = {minV - 0.005:.3f} or lower in Cell 4.")
    if maxQ > Qline_max_pu:
        print(f"  → Max Q-flow {maxQ:.4f} exceeds Qline_max_pu={Qline_max_pu}.")
        print(f"    Try: set Qline_max_pu = {maxQ*1.1:.2f} or higher in Cell 4.")
    raise RuntimeError("Infeasible — adjust parameters in Cell 4 per diagnosis above.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  OPTIONAL: Feasibility relaxation study
#  Run ONLY if the main MILP is infeasible.
#  Finds the tightest Vmin at which the base-case LP (no DER) is feasible.
# ═══════════════════════════════════════════════════════
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Reals,
    Constraint, Objective, minimize, value
)
from pyomo.opt import SolverFactory, TerminationCondition as TC

def base_case_LP(Vmin_try, Vmax_try=1.05, Qlim=1.5, Plim=1.5, verbose=False):
    """LP: no DER, check if network P+Q balance + voltage is feasible.
    Slack bus excluded from P/Q balance (standard LinDistFlow convention).
    Uses load_solutions=False to avoid NoFeasibleSolutionError on new Pyomo.
    """
    mm = ConcreteModel()
    mm.N = Set(initialize=list(range(1, nb+1)))
    mm.L = Set(initialize=list(branches.keys()))
    mm.T = Set(initialize=T)
    mm.Pd    = Param(mm.N, initialize={i: bus_load[i][0]/Sbase_kW   for i in range(1,nb+1)}, within=NonNegativeReals)
    mm.Qd    = Param(mm.N, initialize={i: bus_load[i][1]/Sbase_kVAr for i in range(1,nb+1)}, within=NonNegativeReals)
    mm.lmult = Param(mm.T, initialize={t: float(load_mult[t]) for t in T})
    mm.R     = Param(mm.L, initialize={l: branches[l][2] for l in branches})
    mm.X     = Param(mm.L, initialize={l: branches[l][3] for l in branches})

    mm.Pflow = Var(mm.L, mm.T, within=Reals)
    mm.Qflow = Var(mm.L, mm.T, within=Reals)
    mm.V     = Var(mm.N, mm.T, within=Reals)
    mm.LS    = Var(mm.N, mm.T, within=NonNegativeReals)

    mm.c_v_lo  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.V[n,t] >= Vmin_try)
    mm.c_v_hi  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.V[n,t] <= Vmax_try)
    mm.c_slack = Constraint(mm.T, rule=lambda mm,t: mm.V[1,t] == 1.0)
    mm.c_plim  = Constraint(mm.L, mm.T, rule=lambda mm,l,t: (-Plim, mm.Pflow[l,t], Plim))
    mm.c_qlim  = Constraint(mm.L, mm.T, rule=lambda mm,l,t: (-Qlim, mm.Qflow[l,t], Qlim))

    def vdrop(mm, l, t):
        i = from_bus[l]; j = to_bus[l]
        return mm.V[j,t] == mm.V[i,t] - mm.R[l]*mm.Pflow[l,t] - mm.X[l]*mm.Qflow[l,t]
    mm.c_vdrop = Constraint(mm.L, mm.T, rule=vdrop)

    # Power balance — SKIP slack bus (bus 1); grid supplies freely
    def pb(mm, n, t):
        if n == 1: return Constraint.Skip
        Pin  = sum(mm.Pflow[l,t] for l in in_lines[n])
        Pout = sum(mm.Pflow[l,t] for l in out_lines[n])
        return Pin - Pout == -(mm.Pd[n]*mm.lmult[t] - mm.LS[n,t])
    mm.c_pbal = Constraint(mm.N, mm.T, rule=pb)

    def qb(mm, n, t):
        if n == 1: return Constraint.Skip
        Qin  = sum(mm.Qflow[l,t] for l in in_lines[n])
        Qout = sum(mm.Qflow[l,t] for l in out_lines[n])
        return Qin - Qout == -mm.Qd[n]*mm.lmult[t]
    mm.c_qbal = Constraint(mm.N, mm.T, rule=qb)

    mm.c_ls  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.LS[n,t] <= mm.Pd[n]*mm.lmult[t])
    mm.obj   = Objective(expr=sum(mm.LS[n,t] for n in mm.N for t in mm.T), sense=minimize)

    opt2 = SolverFactory(solver_name)
    # load_solutions=False avoids NoFeasibleSolutionError on new Pyomo
    res2 = opt2.solve(mm, tee=False, load_solutions=False)
    tc2  = res2.solver.termination_condition
    ok   = tc2 in [TC.optimal, TC.feasible]
    ens  = None
    if ok:
        mm.solutions.load_from(res2)
        ens = value(mm.obj)
    if verbose:
        print(f"  Vmin={Vmin_try:.2f} → {'FEASIBLE' if ok else 'INFEASIBLE'}"
              + (f", ENS={ens*Sbase_kW:.1f} kWh" if ens is not None else ""))
    return ok, ens

print("Scanning minimum feasible Vmin (base-case LP, no DER):")
for vmin_try in [0.95, 0.93, 0.90, 0.85, 0.80]:
    ok, ens = base_case_LP(vmin_try, verbose=True)
    if ok:
        print(f"  → Recommended: set Vmin ≤ {vmin_try:.2f} in Cell 4")
        break
else:
    print("  Network infeasible even at Vmin=0.80 — check data.")


## 4. Results — Siting, Sizing & Hosting Capacity

**Hosting Capacity (HC)** is the total installable DER capacity without violating voltage/thermal constraints:

$$HC_{PV} = \sum_{n \in \mathcal{N}_{PV}} P^{PV,max}_n \quad [\text{kW}]$$

$$HC_{BESS,E} = \sum_{n \in \mathcal{N}_{B}} E^{max}_n \quad [\text{kWh}]$$


In [ ]:
# ─── Siting ───────────────────────────────────
pv_sites   = [n for n in m.N if value(m.xPV[n]) > 0.5]
bess_sites = [n for n in m.N if value(m.xB[n])  > 0.5]

print("=== PV SITING ===")
print("PV buses:", pv_sites)
print("\n=== BESS SITING ===")
print("BESS buses:", bess_sites)

# ─── Sizing ───────────────────────────────────
print("\n=== PV SIZING ===")
HC_PV = 0.0
for n in pv_sites:
    kW = value(m.PPVmax[n]) * Sbase_kW
    HC_PV += kW
    print(f"  Bus {n:2d}: PPVmax = {kW:8.1f} kW")
print(f"  → Total HC_PV = {HC_PV:.1f} kW")

print("\n=== BESS SIZING ===")
HC_E = 0.0; HC_Pch = 0.0; HC_Pdis = 0.0
for n in bess_sites:
    Ekwh  = value(m.Emax[n])    * Sbase_kW
    Pch   = value(m.Pchmax[n])  * Sbase_kW
    Pdis  = value(m.Pdismax[n]) * Sbase_kW
    soc0  = value(m.E_init[n,0]) / max(value(m.Emax[n]), 1e-9)  # day 0 initial
    HC_E   += Ekwh; HC_Pch += Pch; HC_Pdis += Pdis
    E_rated_kWh = Ekwh / (SOC_max_frac - SOC_min_frac)  # usable / 0.70
    print(f"  Bus {n:2d}: Emax(usable)={Ekwh:7.1f} kWh  Erated={E_rated_kWh:7.1f} kWh | "
          f"Pch={Pch:6.1f} kW Pdis={Pdis:6.1f} kW")
print(f"  → Total HC_BESS_E    = {HC_E:.1f} kWh")
print(f"  → Total HC_BESS_Pch  = {HC_Pch:.1f} kW")
print(f"  → Total HC_BESS_Pdis = {HC_Pdis:.1f} kW")


## 5. Operational Checks — Voltage Profile & SOC

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

# ── Voltage profile: worst hour across all days ──────────────────
V_all = np.array([
    [value(m.V[n, d, t]) for n in range(1, nb+1)]
    for d in D for t in T24
])  # shape (7*24, 33)
V_worst = V_all.min(axis=0)   # worst voltage per bus across all day-hours

print("Voltage check — worst case across all 7 days x 24 hours:")
print(f"  Global min V = {V_worst.min():.4f} pu at bus {V_worst.argmin()+1}")
buses_below = np.where(V_worst < Vmin)[0] + 1
if len(buses_below):
    print(f"  Buses below Vmin={Vmin}: {buses_below.tolist()}")
else:
    print(f"  No under-voltage violations across all {N_DAYS} days x 24 hours")

# ── Bar chart: worst-case voltage per bus ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

ax = axes[0]
colors = ['#C0392B' if v < Vmin else ('#E8A020' if v < 0.95 else '#2E6FA3')
          for v in V_worst]
ax.bar(range(1, nb+1), V_worst, color=colors, alpha=0.80, width=0.8)
ax.axhline(Vmin, color='#C0392B', ls='--', lw=1.2, label=f'Vmin={Vmin}')
ax.axhline(Vmax, color='#27AE60', ls='--', lw=1.2, label=f'Vmax={Vmax}')
ax.set_xlabel('Bus number'); ax.set_ylabel('Voltage (pu)')
ax.set_title('Worst-case bus voltage across all 7 days')
ax.set_xlim(0.5, nb+0.5); ax.set_ylim(0.86, 1.08)
ax.legend(fontsize=9)

# ── Line plot: voltage profile for each day at peak hour ────────
ax2 = axes[1]
peak_hour = int(np.array([load_7d[d].max() for d in D]).argmax() * 24 +
               load_7d.reshape(N_DAYS, 24).max(axis=1).argmax() * 0)
day_colors = ['#1A4A7A','#2E6FA3','#3A8FC4','#2E85A0',
              '#1D6B88','#4A90A4','#6AAABA']
for d in D:
    t_peak_d = int(load_7d[d].argmax())
    V_day = [value(m.V[n, d, t_peak_d]) for n in range(1, nb+1)]
    ax2.plot(range(1, nb+1), V_day, color=day_colors[d], lw=1.3, alpha=0.85,
             label=f'{DAY_CONFIG[d][0][:3]} t={t_peak_d}h')
ax2.axhline(Vmin, color='#C0392B', ls='--', lw=1.2, label=f'Vmin={Vmin}')
ax2.set_xlabel('Bus number'); ax2.set_ylabel('Voltage (pu)')
ax2.set_title('Voltage profile at each day peak-load hour')
ax2.set_xlim(0.5, nb+0.5); ax2.set_ylim(0.86, 1.08)
ax2.legend(fontsize=7.5, ncol=2)

plt.tight_layout()
plt.savefig('fig_voltage_7day.png', dpi=200, bbox_inches='tight')
plt.show()

# ── SOC profiles: all days for each BESS site ───────────────────
if bess_sites:
    fig2, axes2 = plt.subplots(1, len(bess_sites),
                                figsize=(5*len(bess_sites), 3.5), squeeze=False)
    for col, n in enumerate(bess_sites):
        ax = axes2[0][col]
        Emax_val = value(m.Emax[n])
        for d in D:
            E_vals = [value(m.E[n, d, t]) for t in T24]
            soc = [SOC_min_frac + (e / Emax_val) * (SOC_max_frac - SOC_min_frac)
                   for e in E_vals]
            ax.plot(T24, soc, color=day_colors[d], lw=1.3, alpha=0.85,
                    label=DAY_CONFIG[d][0][:3])
        ax.axhline(SOC_min_frac, color='red',   ls='--', lw=1, label='SOC_min=0.20')
        ax.axhline(SOC_max_frac, color='green', ls='--', lw=1, label='SOC_max=0.90')
        ax.set_title(f'BESS @ Bus {n}', fontsize=10)
        ax.set_xlabel('Hour'); ax.set_ylabel('SOC (pu) [0.20-0.90]')
        ax.set_ylim(0.10, 1.00)
        ax.legend(fontsize=7.5, ncol=2)
    fig2.suptitle('BESS State of Charge — All 7 Days', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('fig_soc_7day.png', dpi=200, bbox_inches='tight')
    plt.show()


## 6. Hosting Capacity Summary Table & CSV Export

In [ ]:
rows = []
for n in m.N:
    rows.append({
        "Bus"         : n,
        "xPV"         : int(round(value(m.xPV[n]))),
        "PPVmax_kW"   : round(value(m.PPVmax[n]) * Sbase_kW, 2),
        "xBESS"       : int(round(value(m.xB[n]))),
        "Emax_kWh"    : round(value(m.Emax[n])    * Sbase_kW, 2),
        "Pchmax_kW"   : round(value(m.Pchmax[n])  * Sbase_kW, 2),
        "Pdismax_kW"  : round(value(m.Pdismax[n]) * Sbase_kW, 2),
    })
df = pd.DataFrame(rows)

# Pretty-print active rows
active = df[(df.xPV==1) | (df.xBESS==1)]
print(active.to_string(index=False))

df.to_csv("pv_bess_hosting_capacity_ieee33.csv", index=False)
print("\nSaved: pv_bess_hosting_capacity_ieee33.csv")

print(f"\n{'='*40}")
print(f" HC_PV   total : {HC_PV:.1f} kW")
print(f" HC_BESS total : {HC_E:.1f} kWh  |  {HC_Pch:.1f} kW charge  |  {HC_Pdis:.1f} kW discharge")
print(f" ENS            : {value(m.obj)*Sbase_kW:.2f} kWh/day")
print(f"{'='*40}")


## 7. Standalone Diagnostics (retained from original)
These helper cells work independently of the optimizer — useful for quick checks during debugging.

In [ ]:
# Voltage diagnostics — post-solve or for manual arrays
import numpy as np

def check_voltage(V_arr, Vmin=0.95, Vmax=1.05):
    """Check voltage array for violations."""
    vmin = V_arr.min(); vmax = V_arr.max()
    violated = np.where((V_arr < Vmin) | (V_arr > Vmax))[0]
    print(f"Minimum voltage : {vmin:.4f} pu")
    print(f"Maximum voltage : {vmax:.4f} pu")
    if len(violated):
        print(f"Violated buses  : {(violated+1).tolist()}")
    else:
        print("No voltage violations ✓")
    return violated

# Example
V_example = np.array([1.00, 0.98, 0.96, 0.94, 1.02])
check_voltage(V_example)


In [ ]:
# SOC diagnostics
import numpy as np

def check_soc(SOC_arr, SOCmin=0.20, SOCmax=0.90):
    """Check SOC array for violations."""
    print(f"SOC min = {SOC_arr.min():.3f}  |  SOC max = {SOC_arr.max():.3f}")
    viol = np.where((SOC_arr < SOCmin) | (SOC_arr > SOCmax))[0]
    if len(viol):
        print(f"Violations at t = {viol.tolist()}  ← would be infeasible in optimizer")
    else:
        print("No SOC violations ✓")
    return viol

SOC_example = np.array([0.50, 0.47, 0.45, 0.42, 0.44, 0.60])
check_soc(SOC_example)


In [ ]:
# SOC simulation — verify dynamics match optimizer convention
import numpy as np

def simulate_soc(P_bess, E_rated_kWh, SOC0=0.50, eta_c=0.95, eta_d=0.95, dt=1.0):
    """
    P_bess: array of kW (negative = charge, positive = discharge)
    Returns SOC array (length = len(P_bess)+1, starts at SOC0).
    Convention: consistent with MILP E[t] = E[t-1] + eta_c*Pch - (1/eta_d)*Pdis
    """
    SOC = [SOC0]
    for p in P_bess:
        if p < 0:   # charging
            dSOC = eta_c * (-p) * dt / E_rated_kWh
            SOC.append(SOC[-1] + dSOC)
        else:       # discharging
            dSOC = p * dt / (eta_d * E_rated_kWh)
            SOC.append(SOC[-1] - dSOC)
    return np.array(SOC)

P_example = np.array([-2, -2, 3, 3, -1])   # kW
soc_trace = simulate_soc(P_example, E_rated_kWh=10.0)
print("SOC trace:", np.round(soc_trace, 4))
print("Cycle balance (end - start):", round(soc_trace[-1] - soc_trace[0], 6),
      "← should be 0 for feasible daily cycle")
